In [ ]:
# [步驟 3-2] 多變量離群值偵測 (Multivariate Outlier Detection)
# 原因 (Reason)： 有些樣本在單一變數上正常，但在多維空間組合中卻是異常的 (如身高高但體重極輕)。
# 目的 (Purpose)： 計算馬氏距離 (Mahalanobis D^2) 並除以自由度 (df)，與卡方分佈臨界值 (p<0.001) 進行比較。
# 預期結果 (Result)： 識別出綜合多個變數後，距離中心點過遠的異常樣本。
from IPython.display import HTML
from plotly.subplots import make_subplots
from scipy import stats
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2, kurtosis, levene, skew, ttest_ind
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, LinearRegression
from statsmodels.stats.diagnostic import lilliefors
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
import tqdm

# table 2-1 開始

## read

In [3]:
# [步聚 1] 讀取資料檔案 (Data Loading)
# 原因 (Reason)： 所有的分析都需要基於原始數據進行，必須先將資料載入記憶體。
# 目的 (Purpose)： 從 Excel 檔案讀取 'HBAT_MISSING' 工作表，並將分類變數 (V10-V14) 轉換為 category 型態，以便正確處理。
# 預期結果 (Result)： 建立一個名為 df 的 Pandas DataFrame，包含所有變數的原始資料。
df = pd.read_excel("data.xls", sheet_name = "HBAT_MISSING", na_values = ["NA", "."])
df[["V10", "V11", "V12", "V13", "V14"]] = df[["V10", "V11", "V12", "V13", "V14"]].astype("category")


## table 2-1

In [4]:
# [步驟 2-1] 缺失值統計分析 - Table 2.1
# 原因 (Reason)： 在進行多變量分析前，必須確認資料的完整性，因為缺失值會嚴重影響分析結果的可信度。
# 目的 (Purpose)： 計算每個變數的缺失值數量、缺失比例，以及變數的平均數與標準差。
# 預期結果 (Result)： 產出 Table 2.1 (Summary Statistics)，讓分析者能快速掌握哪些變數缺失情況嚴重。
# get indicators
n_cases = df.count()
mean_val = df.mean(numeric_only = True)
std_val = df.std(numeric_only = True)
miss_num = df.isnull().sum()
miss_pct = (df.isnull().sum() / len(df)) * 100

# get df
summary = pd.DataFrame({
    ("Number of Cases", ""): n_cases,
    ("Mean", ""): mean_val.round(1),
    ("Standard Deviation", ""): std_val.round(2),
    ("Missing Data", "Number"): miss_num,
    ("Missing Data", "Percent"): miss_pct.round(0)
})

summary = summary.reindex(df.columns)
summary = summary.drop("ID", axis = 0, errors = "ignore")
summary.columns = pd.MultiIndex.from_tuples(summary.columns)

print("=== Summary Statistics of Missing Data for original Sample ===")
display(summary)


case_summary = df.isnull().sum(axis = 1).value_counts().sort_index().to_frame("Number of Cases")

case_summary["Percent of Sample"] = (case_summary["Number of Cases"] / len(df)) * 100

case_summary.loc["Total"] = case_summary.sum()

case_summary.index.name = "Number of Missing Data per Case"
case_summary["Percent of Sample"] = case_summary["Percent of Sample"].round(1)

print("=== Summary of Cases ===")
display(case_summary)


=== Summary Statistics of Missing Data for original Sample ===


Number of Cases  Mean Standard Deviation Missing Data        
                                                   Number Percent
V1               49   4.0               0.93           21    30.0
V2               57   1.9               0.88           13    19.0
V3               53   8.1               1.41           17    24.0
V4               63   5.2               1.17            7    10.0
V5               61   2.9               0.78            9    13.0
V6               64   2.6               0.72            6     9.0
V7               61   6.8               1.68            9    13.0
V8               61  46.0               9.36            9    13.0
V9               63   4.8               0.83            7    10.0
V10              68   NaN                NaN            2     3.0
V11              68   NaN                NaN            2     3.0
V12              68   NaN                NaN            2     3.0
V13              69   NaN                NaN            1     1.0
V14              68   NaN                NaN            2     3.0

=== Summary of Cases ===


,Number of Cases,Percent of Sample
Number of Missing Data per Case,,
0,26.0,37.1
1,15.0,21.4
2,19.0,27.1
3,4.0,5.7
7,6.0,8.6
Total,70.0,100.0


## table 2-2

In [5]:
missing_cases = df[df.isnull().any(axis = 1)].copy()

n_missing = missing_cases.isnull().sum(axis = 1)
pct_missing = (n_missing / len(df.columns[1: ])) * 100

patterns = missing_cases.isnull().map(lambda x: "S" if x else "")
patterns["ID"] = missing_cases["ID"]

# 合併所有資訊
table_2_2 = pd.DataFrame({
    "Case": missing_cases["ID"],
    "# Missing": n_missing,
    "% Missing": pct_missing.round(1)
})
table_2_2 = table_2_2.merge(patterns.rename(columns = {"ID": "Case"}), left_on = "Case", right_on = "Case")
table_2_2.set_index("Case", inplace = True)

# 排序
custom_order = [
    205, 202, 250, 255, 269, 238, 240, 253, 256, 259, 260, 228, 246,
    225, 267, 222, 241, 229, 216, 218, 232, 248, 237, 249, 220, 213,
    257, 203, 231, 219, 244, 227, 224, 268, 235, 204, 207, 221, 245,
    233, 261, 210, 263, 214
]
table_2_2 = table_2_2.reindex(custom_order)

print("=== table_2_2 2.2: Patterns of Missing Data by Case ===")
display(table_2_2)


=== table_2_2 2.2: Patterns of Missing Data by Case ===


,# Missing,% Missing,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14
Case,,,,,,,,,,,,,,,,
205,1,7.1,,,S,,,,,,,,,,,
202,2,14.3,S,,S,,,,,,,,,,,
250,2,14.3,S,,S,,,,,,,,,,,
255,2,14.3,S,,S,,,,,,,,,,,
269,2,14.3,S,,S,,,,,,,,,,,
238,1,7.1,S,,,,,,,,,,,,,
240,1,7.1,S,,,,,,,,,,,,,
253,1,7.1,S,,,,,,,,,,,,,
256,1,7.1,S,,,,,,,,,,,,,


## table 2-3

In [6]:
# [步驟 2-1] 缺失值統計分析 - Table 2.1
# 原因 (Reason)： 在進行多變量分析前，必須確認資料的完整性，因為缺失值會嚴重影響分析結果的可信度。
# 目的 (Purpose)： 計算每個變數的缺失值數量、缺失比例，以及變數的平均數與標準差。
# 預期結果 (Result)： 產出 Table 2.1 (Summary Statistics)，讓分析者能快速掌握哪些變數缺失情況嚴重。

# 1. 建立缺失模式 (這次我們用 tuple 來存變數名稱，方便後續比對)
# 例如: ("V1", "V3") 代表該列缺了 V1 和 V3
# 修改第 1 步：產生 Pattern 時強制排序
def get_pattern_tuple(row):
    return tuple(row.index[row.isnull()].sort_values())

# 產生每個案例的模式 tuple
pattern_tuples = df.apply(get_pattern_tuple, axis=1)

# 2. 統計每種模式的次數
pattern_counts = pattern_tuples.value_counts()

# 3. 建立表格
table_2_3 = pd.DataFrame({"Number of Cases": pattern_counts})


# 4. 產生 "X" 標記
# 修正：不要用 [1:]，改用列表推導式排除 'id'
# 這樣無論 id 是 index 還是 column，或者根本沒有 id，V1 都不會被誤刪
cols_to_show = [c for c in df.columns if c != 'id']

# 建立空表格，欄位包含 V1 ~ V14
pattern_display = pd.DataFrame("", index=table_2_3.index, columns=cols_to_show)

# 填入 X
for pattern in table_2_3.index:
    # 確保只填入存在的欄位 (防呆)
    valid_cols = [col for col in pattern if col in pattern_display.columns]
    pattern_display.loc[[pattern], valid_cols] = "X"

table_2_3 = pd.concat([table_2_3, pattern_display], axis=1)

# 5. 計算 Number of Complete Cases...
def calc_complete_cases(pattern):
    # pattern 是一個 tuple，例如 ("V1", "V3")
    # 移除這些欄位後，計算剩下的完整案例數
    return len(df.drop(columns=list(pattern)).dropna())

table_2_3["Number of Complete Cases..."] = [calc_complete_cases(p) for p in table_2_3.index]

# 這是依照你提供的圖片 (Table 2.3) 觀察出的順序
# 請注意：如果你的資料中沒有出現某個模式，Pandas 會自動忽略它
custom_order = [
    (),                                     # 1. 沒缺
    ("V3", ),                                # 2.
    ("V1", "V3"),                           # 3.
    ("V1", ),                                # 4.
    ("V1", "V4"),                           # 5.
    ("V4", ),                                # 6.
    ("V3", "V4"),                           # 7.
    ("V3", "V5"),                           # 8.
    ("V4", "V5"),                           # 9.
    ("V1", "V5"),                           # 10.
    ("V1", "V2"),                           # 11.
    ("V2", ),                                # 12.
    ("V2", "V3"),                           # 13.
    ("V2", "V7"),                           # 14.
    ("V7", ),                                # 15.
    ("V7", "V8"),                           # 16.
    ("V8", ),                                # 17.
    ("V2", "V8"),                           # 18.
    ("V1", "V2", "V8"),                     # 19.
    ("V9", ),                                # 20.
    ("V5", "V9"),                           # 21.
    ("V1", "V3", "V7"),                     # 22.
    ("V1", "V3", "V4"),                     # 23.
    ("V1", "V3", "V5", "V8", "V12", "V13"), # 24.
    ("V2", "V3", "V5", "V6", "V9", "V12", "V13"), # 25.
    ("V2", "V3", "V6", "V7", "V8", "V9", "V11"),  # 26.
    ("V3", "V4", "V5", "V6", "V7", "V8", "V9"),   # 27.
    ("V2", "V3", "V4", "V5", "V6", "V7", "V9"),   # 28.
    ("V1", "V3", "V4", "V5", "V6", "V9", "V12")   # 29.
]

# 使用 reindex 強制排序
# 為了避免資料中有新模式不在列表裡而被丟掉，我們把沒在列表裡的加在後面
existing_patterns = table_2_3.index.tolist()
remaining_patterns = [p for p in existing_patterns if p not in custom_order]
final_order = [p for p in custom_order if p in existing_patterns] + remaining_patterns

table_2_3 = table_2_3.reindex(final_order)

# 6. 加上雙層表頭
new_cols = []
for col in table_2_3.columns:
    if col == "Number of Cases":
        new_cols.append(("Number<br>of Cases", ""))
    elif col == "Number of Complete Cases...":
        new_cols.append(("Number of Complete<br>Cases if Variables<br>Missing in Pattern Are<br>Not Used", ""))
    else:
        new_cols.append(("Missing Data Patterns", col))

table_2_3.columns = pd.MultiIndex.from_tuples(new_cols)
table_2_3.index = [""] * len(table_2_3)

print("=== Table 2.3: Missing Data Patterns Summary (Custom Sort) ===")
table_2_3 = table_2_3.to_html(escape=False)
display(HTML(table_2_3))


=== Table 2.3: Missing Data Patterns Summary (Custom Sort) ===


## table 2-4

In [7]:
# [步驟 2-1] 缺失值統計分析 - Table 2.1
# 原因 (Reason)： 在進行多變量分析前，必須確認資料的完整性，因為缺失值會嚴重影響分析結果的可信度。
# 目的 (Purpose)： 計算每個變數的缺失值數量、缺失比例，以及變數的平均數與標準差。
# 預期結果 (Result)： 產出 Table 2.1 (Summary Statistics)，讓分析者能快速掌握哪些變數缺失情況嚴重。

# 1. 建立縮減後的資料集
# (1) 刪除缺失值過多的 6 個案例 (缺失數 >= 7 的人)
cases_to_keep = df.isnull().sum(axis=1) < 7
df_reduced = df[cases_to_keep].copy()

# (2) 刪除變數 V1 和 id (如果存在)
cols_to_drop = ['V1']
if 'ID' in df_reduced.columns:
    cols_to_drop.append('ID')

df_reduced = df_reduced.drop(columns=cols_to_drop, errors='ignore')

var_summary_reduced = pd.DataFrame()
var_summary_reduced['Number of Cases'] = df_reduced.count()
var_summary_reduced['Mean'] = df_reduced.mean(numeric_only=True).round(1)
var_summary_reduced['Standard Deviation'] = df_reduced.std(numeric_only=True).round(2)
var_summary_reduced['Missing Number'] = df_reduced.isnull().sum()
var_summary_reduced['Missing Percent'] = (df_reduced.isnull().sum() / len(df_reduced) * 100).round(0).astype(int)

# 移除 id 列 (如果 id 已經變成 index，上面 drop columns 刪不到，這裡再刪一次保險)
if 'id' in var_summary_reduced.index:
    var_summary_reduced = var_summary_reduced.drop('id')

# 建立雙層表頭
new_cols = []
for col in var_summary_reduced.columns:
    if 'Missing' in col:
        new_cols.append(('Missing Data', col.replace('Missing ', '')))
    else:
        new_cols.append((col, ''))
var_summary_reduced.columns = pd.MultiIndex.from_tuples(new_cols)

print("=== Table 2.4 Part 1: Variable Summary ===")
display(var_summary_reduced)

missing_per_case = df_reduced.isnull().sum(axis=1)
case_dist = missing_per_case.value_counts().sort_index().to_frame('Number of Cases')
case_dist['Percent of Sample'] = (case_dist['Number of Cases'] / len(df_reduced) * 100).round(0).astype(int)
case_dist.loc['Total'] = case_dist.sum()
case_dist.loc['Total', 'Percent of Sample'] = 100
case_dist.index.name = 'Number of Missing Data per Case'

print("\n=== Table 2.4 Part 2: Summary of Cases ===")
display(case_dist)


=== Table 2.4 Part 1: Variable Summary ===


Number of Cases  Mean Standard Deviation Missing Data        
                                                   Number Percent
V2               54   1.9               0.86           10      16
V3               50   8.1               1.32           14      22
V4               60   5.1               1.19            4       6
V5               59   2.8               0.75            5       8
V6               63   2.6               0.72            1       2
V7               60   6.8               1.68            4       6
V8               60  46.0               9.42            4       6
V9               60   4.8               0.82            4       6
V10              64   NaN                NaN            0       0
V11              64   NaN                NaN            0       0
V12              64   NaN                NaN            0       0
V13              64   NaN                NaN            0       0
V14              64   NaN                NaN            0       0


=== Table 2.4 Part 2: Summary of Cases ===


,Number of Cases,Percent of Sample
Number of Missing Data per Case,,
0,32,50
1,18,28
2,14,22
Total,64,100


## table 2-5

In [8]:
# === Table 2.5: Assessing the Randomness of Missing Data (Final Corrected) ===

# 1. 定義變數
grouping_vars = ['V2', 'V3', 'V4', 'V5', 'V7', 'V8', 'V9']
test_vars = ['V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']

results = []

for g_var in grouping_vars:
    # 分組
    mask_missing = df_reduced[g_var].isnull()
    group_valid = df_reduced[~mask_missing]
    group_missing = df_reduced[mask_missing]

    # 初始化 6 列
    row_t = {'Group': g_var, 'Stat': 't value'}
    row_p = {'Group': g_var, 'Stat': 'Significance'}
    row_n_val = {'Group': g_var, 'Stat': 'Number of cases (valid data)'}
    row_n_mis = {'Group': g_var, 'Stat': 'Number of cases (missing data)'}
    row_m_val = {'Group': g_var, 'Stat': 'Mean of cases (valid data)'}
    row_m_mis = {'Group': g_var, 'Stat': 'Mean of cases (missing data)'}

    for t_var in test_vars:
        # 取出數據
        data_valid = group_valid[t_var].dropna()
        data_missing = group_missing[t_var].dropna()

        # 1. 計算個數 (N)
        n1 = len(data_valid)
        n2 = len(data_missing)
        row_n_val[t_var] = n1
        row_n_mis[t_var] = n2

        # 2. 計算平均數 (Mean)
        if n1 > 0: row_m_val[t_var] = round(data_valid.mean(), 1)
        if n2 > 0: row_m_mis[t_var] = round(data_missing.mean(), 1)

        # 3. 進行 T 檢定 (對角線不做，樣本不足不做)
        if g_var != t_var and n1 > 1 and n2 > 1:
            # 使用 Welch's t-test (equal_var=False) 以符合 SPSS 常見設定
            t_stat, p_val = ttest_ind(data_valid, data_missing, equal_var=False)
            row_t[t_var] = round(t_stat, 1)
            row_p[t_var] = round(p_val, 3)
        else:
            # 對角線或無法計算時，填入 '.'
            row_t[t_var] = '.'
            row_p[t_var] = '.'

    results.extend([row_t, row_p, row_n_val, row_n_mis, row_m_val, row_m_mis])

# 格式化輸出
table_2_5 = pd.DataFrame(results)
table_2_5 = table_2_5.set_index(['Group', 'Stat'])
table_2_5 = table_2_5[test_vars] # 確保欄位順序

print("=== Table 2.5: Assessing the Randomness of Missing Data ===")
# 顯示時把 NaN 換成 '.'
display(table_2_5.fillna('.'))


=== Table 2.5: Assessing the Randomness of Missing Data ===


V2     V3     V4     V5      V6  \
Group Stat                                                                 
V2    t value                             .    0.7   -2.2   -4.2  -2.400   
      Significance                        .  0.528  0.044  0.001   0.034   
      Number of cases (valid data)       54     42     50     49  53.000   
      Number of cases (missing data)      0      8     10     10  10.000   
      Mean of cases (valid data)        1.9    8.2    5.0    2.7   2.500   
      Mean of cases (missing data)        .    7.9    5.9    3.5   3.100   
V3    t value                           1.4      .    1.1    2.0   0.200   
      Significance                     0.18      .  0.286  0.066   0.818   
      Number of cases (valid data)       42     50     48     47  49.000   
      Number of cases (missing data)     12      0     12     12  14.000   
      Mean of cases (valid data)        2.0    8.1    5.2    2.9   2.600   
      Mean of cases (missing data)      1.6      .    4.8    2.4   2.600   
V4    t value                           2.6   -0.3      .    0.2   1.400   
      Significance                    0.046  0.785      .  0.888   0.249   
      Number of cases (valid data)       50     48     60     55  59.000   
      Number of cases (missing data)      4      2      0      4   4.000   
      Mean of cases (valid data)        1.9    8.1    5.1    2.8   2.600   
      Mean of cases (missing data)      1.3    8.4      .    2.8   2.200   
V5    t value                          -0.3    0.8    0.4      .  -0.900   
      Significance                    0.749  0.502  0.734      .   0.423   
      Number of cases (valid data)       49     47     55     59  58.000   
      Number of cases (missing data)      5      3      5      0   5.000   
      Mean of cases (valid data)        1.9    8.2    5.2    2.8   2.600   
      Mean of cases (missing data)      2.0    7.1    5.0      .   2.900   
V7    t value                           0.9    0.2   -2.1    0.9  -1.500   
      Significance                     0.44  0.864  0.118  0.441   0.193   
      Number of cases (valid data)       51     47     56     55  59.000   
      Number of cases (missing data)      3      3      4      4   4.000   
      Mean of cases (valid data)        1.9    8.1    5.1    2.9   2.600   
      Mean of cases (missing data)      1.5    8.0    6.2    2.6   2.900   
V8    t value                          -1.4    2.2   -1.1   -0.9  -1.800   
      Significance                    0.384  0.101  0.326  0.401   0.149   
      Number of cases (valid data)       52     46     56     55  59.000   
      Number of cases (missing data)      2      4      4      4   4.000   
      Mean of cases (valid data)        1.9    8.3    5.1    2.8   2.600   
      Mean of cases (missing data)      3.0    6.6    5.6    3.1   3.000   
V9    t value                           0.8   -2.1    2.5    2.7   1.300   
      Significance                    0.463  0.235  0.076  0.056   0.302   
      Number of cases (valid data)       50     48     56     55  60.000   
      Number of cases (missing data)      4      2      4      4   3.000   
      Mean of cases (valid data)        1.9    8.1    5.2    2.9   2.600   
      Mean of cases (missing data)      1.6    9.2    4.0    2.1   2.200   

                                         V7     V8     V9  
Group Stat                                                 
V2    t value                          -1.2   -1.1   -1.2  
      Significance                     0.26  0.318  0.233  
      Number of cases (valid data)       51     52     50  
      Number of cases (missing data)      9      8     10  
      Mean of cases (valid data)        6.7   45.5    4.8  
      Mean of cases (missing data)      7.4   49.2    5.0  
V3    t value                           0.0    1.9    0.9  
      Significance                    0.965  0.073  0.399  
      Number of cases (valid data)       47     46     48  
      Number of cases (missing data)

## table 2-6(無法完全對準)

In [9]:
# [步驟 2-3] 缺失值填補 (Imputation)
# 原因 (Reason)： 多數多變量算法無法處理 NaN，直接刪除樣本會損失太多資訊，故需進行填補。
# 目的 (Purpose)： 使用 'IterativeImputer' (多變量插補法)，利用變數間的相關性來預測並填補缺失值。
# 預期結果 (Result)： 獲得一個完全沒有缺失值的完整資料集，可供後續分析使用。


# 1. 資料準備：納入輔助變數
df_impute = df.copy()

# 確保 ID 是 Index
if 'id' in df_impute.columns:
    df_impute = df_impute.set_index('id')
elif 'ID' in df_impute.columns:
    df_impute = df_impute.set_index('ID')

# 刪除嚴重缺失案例 (< 7)
cases_to_keep = df_impute.isnull().sum(axis=1) < 7
df_impute = df_impute[cases_to_keep]

# 排除 V1
if 'V1' in df_impute.columns:
    df_impute = df_impute.drop(columns=['V1'])

# 數值變數 (主要分析對象)
cols_numeric = ['V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']
# 輔助變數 (V10~V14)
cols_aux = ['V10', 'V11', 'V12', 'V13', 'V14']

# 建立訓練用 DataFrame
train_data = df_impute[cols_numeric].copy()

# 將輔助變數轉為數值 (處理 category -> code)
for col in cols_aux:
    if col in df_impute.columns:
        if df_impute[col].dtype.name == 'category':
            # 將 category 轉為整數編碼 (0, 1, 2...)，缺失值設為 NaN
            train_data[col] = df_impute[col].cat.codes.replace(-1, np.nan)
        else:
            train_data[col] = df_impute[col]

# 2. 定義插補方法
# Mean: 只針對該變數本身做平均
model_mean = SimpleImputer(strategy='mean')

# Regression / MI: 使用所有變數 (含輔助變數) 做 Bayesian Ridge
# BayesianRidge 符合書中 Bayesian estimation 描述
model_reg_det = IterativeImputer(estimator=BayesianRidge(), sample_posterior=False, random_state=0)
model_reg_err = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=42)
# Python 無直接 EM，用 MICE (w/o posterior) 模擬，它也是一種迭代估計
model_em_proxy = IterativeImputer(estimator=BayesianRidge(), sample_posterior=False, random_state=999, max_iter=50)

# 執行插補並儲存結果
results_data = {}

# (1) Mean Subst (只對數值變數)
df_mean_filled = df_impute[cols_numeric].copy()
df_mean_filled = pd.DataFrame(model_mean.fit_transform(df_mean_filled),
                              columns=cols_numeric, index=df_mean_filled.index)
results_data['Mean subst'] = df_mean_filled

# (2) Regression / EM / MI (使用含輔助變數的 train_data)
imputers = {
    'Reg w/o err': model_reg_det,
    'Reg w/ err': model_reg_err,
    'EM': model_em_proxy
}
# 加入 MI 1~5
for i in range(1, 6):
    imputers[f'Imputation {i}'] = IterativeImputer(estimator=BayesianRidge(),
                                                   sample_posterior=True,
                                                   random_state=i)

for name, model in imputers.items():
    # 訓練並轉換 (包含輔助變數)
    filled_matrix = model.fit_transform(train_data)
    df_filled = pd.DataFrame(filled_matrix, columns=train_data.columns, index=train_data.index)
    # 只取回主要數值變數
    results_data[name] = df_filled[cols_numeric]

# 3. 填表
target_ids = [202, 204, 250, 227, 237, 203, 213, 257]
rows = []

for pid in target_ids:
    if pid not in df_impute.index: continue

    for var in ['V2', 'V3']:
        val = df_impute.loc[pid, var]
        # 基本資料
        row = {'ID': pid, 'Variable': var, 'Values': val if pd.notna(val) else 'Missing'}

        # 填入插補值
        for method_name, df_res in results_data.items():
            if pd.isna(val):
                row[method_name] = round(df_res.loc[pid, var], 1)
            else:
                row[method_name] = '' # 原始值存在則留白

        rows.append(row)

# 4. 顯示
if rows:
    table_2_6 = pd.DataFrame(rows).set_index(['ID', 'Variable'])
    print("=== Table 2.6: Imputed Values Comparison (With Auxiliary Variables) ===")
    display(table_2_6)
else:
    print("錯誤：沒有產生任何資料列，請檢查 target_ids 是否正確。")


=== Table 2.6: Imputed Values Comparison (With Auxiliary Variables) ===


Values Mean subst Reg w/o err Reg w/ err   EM Imputation 1  \
ID  Variable                                                                
202 V2            0.4                                                       
    V3        Missing        8.1         8.5       11.1  8.5          9.1   
204 V2            1.5                                                       
    V3        Missing        8.1         6.9        8.8  6.9          9.1   
250 V2            3.7                                                       
    V3        Missing        8.1         8.2        7.8  8.2          8.9   
227 V2        Missing        1.9         3.3        1.4  3.3          3.6   
    V3            5.7                                                       
237 V2        Missing        1.9         3.9        3.3  3.9          3.8   
    V3            7.4                                                       
203 V2        Missing        1.9         3.5        1.1  3.5          4.8   
    V3            9.1                                                       
213 V2        Missing        1.9         3.4        3.9  3.4          0.7   
    V3        Missing        8.1         6.7        7.9  6.7          5.9   
257 V2        Missing        1.9         3.5        5.2  3.5          5.1   
    V3        Missing        8.1         6.8        6.8  6.8          6.8   

             Imputation 2 Imputation 3 Imputation 4 Imputation 5  
ID  Variable                                                      
202 V2                                                            
    V3                8.0          6.0          7.5          9.4  
204 V2                                                            
    V3                7.3          8.5          6.0          7.0  
250 V2                                                            
    V3                8.3          7.6          6.9          5.9  
227 V2                3.2          3.4          3.6          2.9  
    V3                                                            
237 V2                3.4          4.4          2.1          4.3  
    V3                                                            
203 V2                5.3          3.0          3.8          1.9  
    V3                                                            
213 V2                3.1          3.1          1.8          2.9  
    V3                5.0          7.0          7.9          8.0  
257 V2                2.4          3.5          3.3          5.9  
    V3                7.0          8.1          7.1          7.1

## table 2-7(也無法完全對準)

In [10]:
# [步驟 2-3] 缺失值填補 (Imputation)
# 原因 (Reason)： 多數多變量算法無法處理 NaN，直接刪除樣本會損失太多資訊，故需進行填補。
# 目的 (Purpose)： 使用 'IterativeImputer' (多變量插補法)，利用變數間的相關性來預測並填補缺失值。
# 預期結果 (Result)： 獲得一個完全沒有缺失值的完整資料集，可供後續分析使用。

# 1. 資料準備
df_base = df.copy()
if 'id' in df_base.columns: df_base = df_base.set_index('id')
elif 'ID' in df_base.columns: df_base = df_base.set_index('ID')

# 縮減樣本 (N=64)
keep_mask = df_base.isnull().sum(axis=1) < 7
df_reduced = df_base[keep_mask].copy()
if 'V1' in df_reduced.columns: df_reduced = df_reduced.drop(columns=['V1'])

# 定義變數
target_vars = ['V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']
aux_vars = ['V10', 'V11', 'V12', 'V13', 'V14']

# 準備兩種資料集
# (A) 僅主要變數 (用於大部分方法)
df_numeric = df_reduced[target_vars].copy()

# (B) 含輔助變數 (僅用於 MI with Aux)
df_with_aux = df_reduced[target_vars + aux_vars].copy()
for col in aux_vars:
    if col in df_with_aux.columns:
        if df_with_aux[col].dtype.name == 'category':
            df_with_aux[col] = df_with_aux[col].cat.codes.replace(-1, np.nan)

results = []

def add_result(method_name, df_in):
    mean_vals = df_in[target_vars].mean()
    std_vals = df_in[target_vars].std()
    for var in target_vars:
        results.append({'Method': method_name, 'Stat': 'Mean', 'Variable': var, 'Value': mean_vals[var]})
        results.append({'Method': method_name, 'Stat': 'Standard Deviation', 'Variable': var, 'Value': std_vals[var]})

# --- 方法實作 ---

# 1. Complete Case (Listwise)
add_result('Complete Case (Listwise)', df_numeric.dropna())

# 2. All Available (Pairwise) - 應與圖片 8.130 接近
add_result('All Available (Pairwise)', df_numeric)

# 3. Mean Substitution - 應與 Pairwise Mean 相同
imp_mean = SimpleImputer(strategy='mean')
df_mean = pd.DataFrame(imp_mean.fit_transform(df_numeric), columns=target_vars, index=df_numeric.index)
add_result('Mean Substitution', df_mean)

# 4. Regression w/o error (Deterministic OLS) - 只用 df_numeric
# 改回使用 LinearRegression (OLS) 且不加輔助變數
imp_reg_det = IterativeImputer(estimator=LinearRegression(), sample_posterior=False, random_state=0)
df_reg_det = pd.DataFrame(imp_reg_det.fit_transform(df_numeric), columns=target_vars, index=df_numeric.index)
add_result('Regression without error', df_reg_det)

# 5. Regression w/ error (Stochastic) - 只用 df_numeric
imp_reg_err = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=42)
df_reg_err = pd.DataFrame(imp_reg_err.fit_transform(df_numeric), columns=target_vars, index=df_numeric.index)
add_result('Regression with error', df_reg_err)

# 6. EM (MICE Proxy) - 只用 df_numeric
imp_em = IterativeImputer(estimator=BayesianRidge(), sample_posterior=False, random_state=999)
df_em = pd.DataFrame(imp_em.fit_transform(df_numeric), columns=target_vars, index=df_numeric.index)
add_result('EM', df_em)

# 7. MI without Aux - 只用 df_numeric
mi_no_aux_list = []
for i in range(1, 6):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=i)
    out = pd.DataFrame(imp.fit_transform(df_numeric), columns=target_vars, index=df_numeric.index)
    mi_no_aux_list.append(out)

avg_leads_no_aux = pd.concat([d.mean() for d in mi_no_aux_list], axis=1).mean(axis=1)
for var in target_vars:
    results.append({'Method': 'Multiple Imputation without auxiliary variables', 'Stat': 'Mean', 'Variable': var, 'Value': avg_leads_no_aux[var]})
    results.append({'Method': 'Multiple Imputation without auxiliary variables', 'Stat': 'Standard Deviation', 'Variable': var, 'Value': np.nan})

# 8. MI WITH Aux - 只有這個使用 df_with_aux
mi_aux_list = []
for i in range(1, 6):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=i)
    # 訓練用含輔助變數，但結果只要 target_vars
    out = pd.DataFrame(imp.fit_transform(df_with_aux), columns=df_with_aux.columns, index=df_with_aux.index)[target_vars]
    mi_aux_list.append(out)

avg_leads_aux = pd.concat([d.mean() for d in mi_aux_list], axis=1).mean(axis=1)
for var in target_vars:
    results.append({'Method': 'Multiple Imputation with auxiliary variables', 'Stat': 'Mean', 'Variable': var, 'Value': avg_leads_aux[var]})
    results.append({'Method': 'Multiple Imputation with auxiliary variables', 'Stat': 'Standard Deviation', 'Variable': var, 'Value': np.nan})

# 9. Individual Imputations (from MI with Aux)
for i, d in enumerate(mi_aux_list):
    add_result(f'Imputation {i+1}', d)

# 輸出
df_res = pd.DataFrame(results)
method_order = [
    'Complete Case (Listwise)', 'All Available (Pairwise)', 'Mean Substitution',
    'Regression without error', 'Regression with error', 'EM',
    'Multiple Imputation without auxiliary variables', 'Multiple Imputation with auxiliary variables',
    'Imputation 1', 'Imputation 2', 'Imputation 3', 'Imputation 4', 'Imputation 5'
]
df_res['Method'] = pd.Categorical(df_res['Method'], categories=method_order, ordered=True)
table_2_7 = df_res.pivot_table(index=['Stat', 'Method'], columns='Variable', values='Value', sort=False, observed=False)
table_2_7 = table_2_7.reindex(['Mean', 'Standard Deviation'], level=0)
table_2_7 = table_2_7[target_vars]

print("=== Table 2.7: Comparing Method Estimates (Optimized) ===")
display(table_2_7.fillna('NC').style.format('{:.3f}'))


=== Table 2.7: Comparing Method Estimates (Optimized) ===


## table 2-8

In [11]:
# [步驟 2-3] 缺失值填補 (Imputation)
# 原因 (Reason)： 多數多變量算法無法處理 NaN，直接刪除樣本會損失太多資訊，故需進行填補。
# 目的 (Purpose)： 使用 'IterativeImputer' (多變量插補法)，利用變數間的相關性來預測並填補缺失值。
# 預期結果 (Result)： 獲得一個完全沒有缺失值的完整資料集，可供後續分析使用。

# --- 1. 資料準備 (與 Table 2.7 保持完全一致) ---
# 確保資料環境乾淨
df_clean = df.copy()
if 'id' in df_clean.columns: df_clean = df_clean.set_index('id')
elif 'ID' in df_clean.columns: df_clean = df_clean.set_index('ID')

# 篩選 Reduced Sample (N=64)
mask = df_clean.isnull().sum(axis=1) < 7
df_reduced_final = df_clean[mask].copy()
if 'V1' in df_reduced_final.columns: df_reduced_final = df_reduced_final.drop(columns=['V1'])

# 變數定義 (這次只關注 V2-V5 的相關性)
target_vars = ['V2', 'V3', 'V4', 'V5'] # Table 2.8 只有這四個
aux_vars = ['V10', 'V11', 'V12', 'V13', 'V14']
# 為了訓練模型，我們需要全部的 V2-V9
train_vars = ['V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']

# (A) 純數值資料 (df_num)
df_num = df_reduced_final[train_vars].copy()

# (B) 含輔助變數資料 (df_encoded) - One Hot Encoded
df_aux_raw = df_reduced_final[train_vars + aux_vars].copy()
df_encoded = pd.get_dummies(df_aux_raw, columns=aux_vars, dummy_na=False, drop_first=True)
df_encoded = df_encoded.astype(float)


# --- 2. 定義儲存結構 ---
results = []
# 目標結構：每個 Row 代表 (RowVariable, Method)，Columns 是 V2, V3, V4, V5

def add_corr_res(method_name, df_in):
    # 計算相關矩陣 (只針對 V2-V5)
    corr_mat = df_in[target_vars].corr()

    # 將每一列 (Row Variable) 加入結果
    for r_var in target_vars:
        row_data = {
            'Target': r_var,
            'Method': method_name
        }
        # 填入該變數與 V2, V3, V4, V5 的相關係數
        for c_var in target_vars:
            row_data[c_var] = corr_mat.loc[r_var, c_var]

        results.append(row_data)

# --- 3. 執行各方法 (邏輯同 Table 2.7) ---

# (1) Listwise
add_corr_res('Complete Case (Listwise)', df_num.dropna())

# (2) Pairwise (Pandas corr 預設就是 Pairwise)
add_corr_res('All Available (Pairwise)', df_num)

# (3) Mean Substitution
df_mean_fix = df_num.copy()
for c in df_mean_fix.columns:
    df_mean_fix[c] = df_mean_fix[c].fillna(df_mean_fix[c].mean())
add_corr_res('Mean Substitution', df_mean_fix)

# (4) Regression w/o Error (OLS)
imp_ols = IterativeImputer(estimator=LinearRegression(), sample_posterior=False, random_state=0)
res_ols = imp_ols.fit_transform(df_encoded)
df_reg_ols = pd.DataFrame(res_ols, columns=df_encoded.columns, index=df_encoded.index)
add_corr_res('Regression without error', df_reg_ols)

# (5) Regression w/ Error (Stochastic)
imp_bayes = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=42)
res_bayes = imp_bayes.fit_transform(df_encoded)
df_reg_stoch = pd.DataFrame(res_bayes, columns=df_encoded.columns, index=df_encoded.index)
add_corr_res('Regression with error', df_reg_stoch)

# (6) EM
imp_em = IterativeImputer(estimator=BayesianRidge(), sample_posterior=False, random_state=999)
res_em = imp_em.fit_transform(df_encoded)
df_em = pd.DataFrame(res_em, columns=df_encoded.columns, index=df_encoded.index)
add_corr_res('EM', df_em)

# (7) MI without Aux (Mean of Correlation Matrices)
corr_mats_no_aux = []
for i in range(1, 6):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=i)
    # 只用 df_num
    out = pd.DataFrame(imp.fit_transform(df_num), columns=train_vars, index=df_num.index)
    corr_mats_no_aux.append(out[target_vars].corr())

# 計算平均矩陣
avg_corr_no_aux = pd.concat([m for m in corr_mats_no_aux]).groupby(level=0).mean()
# 手動加入結果
for r_var in target_vars:
    row_data = {'Target': r_var, 'Method': 'Multiple Imputation without auxiliary variables'}
    for c_var in target_vars:
        row_data[c_var] = avg_corr_no_aux.loc[r_var, c_var]
    results.append(row_data)

# (8) MI WITH Aux
corr_mats_aux = []
for i in range(1, 6):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=i)
    # 用 df_encoded
    res = imp.fit_transform(df_encoded)
    out = pd.DataFrame(res, columns=df_encoded.columns, index=df_encoded.index)
    corr_mats_aux.append(out[target_vars].corr())

# 計算平均矩陣
avg_corr_aux = pd.concat([m for m in corr_mats_aux]).groupby(level=0).mean()
for r_var in target_vars:
    row_data = {'Target': r_var, 'Method': 'Multiple Imputation with auxiliary variables'}
    for c_var in target_vars:
        row_data[c_var] = avg_corr_aux.loc[r_var, c_var]
    results.append(row_data)


# --- 4. 格式化輸出 (靠上對齊版) ---
# ... (DataFrame 排序邏輯同上) ...
row_order = []
for v in ['V2', 'V3', 'V4', 'V5']:
    for m in method_order:
        row_order.append((v, m))
table_2_8 = table_2_8.reindex(row_order)
table_2_8 = table_2_8[['V2', 'V3', 'V4', 'V5']]
print("=== Table 2.8: Comparison of Correlations (Top Aligned) ===")
# 定義 CSS 樣式
styles = [
    # 針對 Index (Variable & Method) 設定靠上對齊
    {'selector': 'th', 'props': [('vertical-align', 'top')]},
    # 針對數據儲存格設定置中
    {'selector': 'td', 'props': [('text-align', 'center')]}
]
# 應用樣式
display(table_2_8.style.format('{:.3f}').set_table_styles(styles))


NameError: name 'table_2_8' is not defined

## table 2-9

In [12]:
# [步驟 2-3] 缺失值填補 (Imputation)
# 原因 (Reason)： 多數多變量算法無法處理 NaN，直接刪除樣本會損失太多資訊，故需進行填補。
# 目的 (Purpose)： 使用 'IterativeImputer' (多變量插補法)，利用變數間的相關性來預測並填補缺失值。
# 預期結果 (Result)： 獲得一個完全沒有缺失值的完整資料集，可供後續分析使用。

# --- 1. 資料準備 (Data Prep) ---
df_clean = df.copy()
if 'id' in df_clean.columns: df_clean = df_clean.set_index('id')
elif 'ID' in df_clean.columns: df_clean = df_clean.set_index('ID')

# Reduced Sample (N=64)
mask = df_clean.isnull().sum(axis=1) < 7
df_reduced_final = df_clean[mask].copy()
if 'V1' in df_reduced_final.columns: df_reduced_final = df_reduced_final.drop(columns=['V1'])

# 定義模型變數
dep_var = 'V9'
indep_vars = ['V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8']
all_vars = indep_vars + [dep_var]
aux_vars = ['V10', 'V11', 'V12', 'V13', 'V14']

# (A) 純數值
df_num = df_reduced_final[all_vars].copy()

# (B) 含輔助變數 (One-Hot)
df_aux_raw = df_reduced_final[all_vars + aux_vars].copy()
df_encoded = pd.get_dummies(df_aux_raw, columns=aux_vars, dummy_na=False, drop_first=True).astype(float)


# --- 2. 輔助函數：執行回歸並提取結果 ---
results = []
cols_order = ['Intercept'] + indep_vars + ['R2']

def run_ols(method_name, df_data):
    # 準備 X, y
    y = df_data[dep_var]
    X = df_data[indep_vars]
    X = sm.add_constant(X) # 加入截距項

    # 執行回歸
    model = sm.OLS(y, X).fit()

    # 提取係數與顯著性
    row_data = {'Method': method_name, 'R2': model.rsquared}

    # 處理截距
    p_const = model.pvalues['const']
    beta_const = model.params['const']
    # 如果 p < .05 則加粗 (Markdown 語法)
    row_data['Intercept'] = f"**{beta_const:.3f}**" if p_const < .05 else f"{beta_const:.3f}"

    # 處理各變數
    for v in indep_vars:
        p_val = model.pvalues[v]
        beta = model.params[v]
        row_data[v] = f"**{beta:.3f}**" if p_val < .05 else f"{beta:.3f}"

    results.append(row_data)

# --- 3. 執行各方法 ---

# (1) Listwise
run_ols("Complete Case (Listwise)", df_num.dropna())

# (2) Pairwise (特殊處理：矩陣運算)
# 計算相關矩陣 (Pairwise Correlation)
corr_mat = df_num.corr()
# 計算標準差與平均數 (Pairwise)
std_vals = df_num.std()
mean_vals = df_num.mean()

# 分割 correation matrix 為 Rxx, Rxy
R_xx = corr_mat.loc[indep_vars, indep_vars].values
R_xy = corr_mat.loc[indep_vars, dep_var].values

# 解標準化係數 Beta* = inv(Rxx) * Rxy
Beta_star = np.linalg.inv(R_xx) @ R_xy

# 轉換回未標準化係數 Beta = Beta* * (SDy / SDx)
SD_y = std_vals[dep_var]
SD_x = std_vals[indep_vars].values
Beta = Beta_star * (SD_y / SD_x)

# 計算截距 Intercept = Mean_y - sum(Beta * Mean_x)
Mean_y = mean_vals[dep_var]
Mean_x = mean_vals[indep_vars].values
Intercept = Mean_y - np.sum(Beta * Mean_x)

# 估算 R2 (Beta* dot Rxy)
R2_pairwise = np.dot(Beta_star, R_xy)

# 儲存結果 (Pairwise 通常難以計算準確的 p-value，這裡只顯示係數)
# 為了跟圖片一致，顯著性部分我們假設跟係數大小有關，先只顯示數值
row_pw = {'Method': 'All Available (Pairwise)', 'R2': R2_pairwise}
row_pw['Intercept'] = f"{Intercept:.3f}"
for i, v in enumerate(indep_vars):
    # 雖然無法精確算 P 值，但我們可以標記強係數 (根據圖片 V3, V4, V5 是顯著的)
    # 這裡純顯示數值
    row_pw[v] = f"{Beta[i]:.3f}"
results.append(row_pw)


# (3) Mean Subst
df_mean_fix = df_num.copy()
for c in df_mean_fix.columns:
    df_mean_fix[c] = df_mean_fix[c].fillna(df_mean_fix[c].mean())
run_ols("Mean Substitution", df_mean_fix)

# (4) Reg w/o Error (OLS)
imp_ols = IterativeImputer(estimator=LinearRegression(), sample_posterior=False, random_state=0)
res_ols = imp_ols.fit_transform(df_encoded)
df_reg_ols = pd.DataFrame(res_ols, columns=df_encoded.columns, index=df_encoded.index)
run_ols("Regression without Error", df_reg_ols)

# (5) Reg w/ Error
imp_bayes = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=42)
res_bayes = imp_bayes.fit_transform(df_encoded)
df_reg_stoch = pd.DataFrame(res_bayes, columns=df_encoded.columns, index=df_encoded.index)
run_ols("Regression with Error", df_reg_stoch)

# (6) EM
imp_em = IterativeImputer(estimator=BayesianRidge(), sample_posterior=False, random_state=999)
res_em = imp_em.fit_transform(df_encoded)
df_em = pd.DataFrame(res_em, columns=df_encoded.columns, index=df_encoded.index)
run_ols("EM", df_em)

# (7) MI with Auxiliary Variables (Rubin's Rules)
models = []
for i in range(1, 6):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=i)
    res = imp.fit_transform(df_encoded)
    df_mi = pd.DataFrame(res, columns=df_encoded.columns, index=df_encoded.index)

    y = df_mi[dep_var]
    X = df_mi[indep_vars]
    X = sm.add_constant(X)
    models.append(sm.OLS(y, X).fit())

# Pool Coefficients (Average)
params_list = [m.params for m in models]
pooled_params = pd.concat(params_list, axis=1).mean(axis=1)

# Pool Standard Errors (Rubin's Rules)
# Within Variance
se_list = [m.bse**2 for m in models]
within_var = pd.concat(se_list, axis=1).mean(axis=1)
# Between Variance
between_var = pd.concat(params_list, axis=1).var(axis=1) # ddof=1 default
# Total Variance
total_var = within_var + (1 + 1/5) * between_var
pooled_se = np.sqrt(total_var)

# Calculate t and p-values
# degrees of freedom (simplified version)
df_mi = 64 - 8 # approx
t_vals = pooled_params / pooled_se
p_vals = stats.t.sf(np.abs(t_vals), df_mi) * 2

# Pool R2
pooled_r2 = np.mean([m.rsquared for m in models])

# Assemble MI Row
row_mi = {'Method': 'Multiple Imputation with Auxiliary Variables', 'R2': pooled_r2}

# Const
p_c = p_vals[0] # const is usually index 0
beta_c = pooled_params['const']
row_mi['Intercept'] = f"**{beta_c:.3f}**" if p_c < .05 else f"{beta_c:.3f}"

for i, v in enumerate(indep_vars):
    idx = i + 1 # shift for const
    p_v = p_vals[idx]
    beta_v = pooled_params[v]
    row_mi[v] = f"**{beta_v:.3f}**" if p_v < .05 else f"{beta_v:.3f}"

results.append(row_mi)


# --- 4. 輸出表格 ---
df_res = pd.DataFrame(results)
ordering = [
    "Complete Case (Listwise)", "All Available (Pairwise)", "Mean Substitution",
    "Regression without Error", "Regression with Error", "EM",
    "Multiple Imputation with Auxiliary Variables"
]
df_res['Method'] = pd.Categorical(df_res['Method'], categories=ordering, ordered=True)
df_res = df_res.set_index('Method')
df_res = df_res[cols_order] # Reorder columns

print("=== Table 2.9: Regression Results (Bold = Significant p<.05) ===")
# 使用 .to_markdown() 如果你想看純文字，或者用 style
display(df_res.style.set_properties(**{'text-align': 'center'}))


=== Table 2.9: Regression Results (Bold = Significant p<.05) ===


,Intercept,V2,V3,V4,V5,V6,V7,V8,R2
Method,,,,,,,,,
Complete Case (Listwise),0.140,-0.204,**0.483**,**0.311**,**0.492**,-0.015,**-0.153**,-0.018,0.808746
All Available (Pairwise),-0.238,-0.195,0.508,0.271,0.823,-0.103,-0.072,-0.037,0.896731
Mean Substitution,0.514,**-0.213**,**0.306**,**0.258**,**0.346**,-0.091,-0.072,0.013,0.714737
Regression without Error,-0.045,-0.149,**0.348**,**0.400**,**0.602**,-0.237,**-0.093**,-0.005,0.793771
Regression with Error,0.676,-0.109,**0.213**,**0.386**,**0.313**,-0.234,-0.076,0.019,0.673900
EM,-0.053,-0.137,**0.357**,**0.364**,**0.575**,-0.225,**-0.080**,-0.004,0.790809
Multiple Imputation with Auxiliary Variables,0.439,-0.136,**0.266**,**0.344**,**0.429**,-0.242,-0.060,0.011,0.719413


# table 2-10 開始

## read

In [9]:
# === 讀取資料檔案 ===
df = pd.read_excel("data.xls", sheet_name = "HBAT", na_values = ["NA", "."])
df[["X1", "X2", "X3", "X4", "X5", "X23"]] = df[["X1", "X2", "X3", "X4", "X5", "X23"]].astype("category")

## fig 2-10

In [5]:
# [步驟 3-3] 離群值偵測總表 - Table 2.10
# 原因 (Reason)： 需要一個總表來整合單變量、雙變量與多變量偵測的結果，以便綜合判斷。
# 目的 (Purpose)： 彙整前面的計算成果，列出每個變數的異常樣本 ID，並標註多變量異常指標 (D2)。
# 預期結果 (Result)： 產出 Table 2.10，做為決定是否刪除或保留特定樣本的依據。

def get_ellipse_coordinates(x, y, confidence=0.95):
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    n = x.size
    cov = np.cov(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    # Eigen decomposition
    vals, vecs = np.linalg.eigh(cov)

    # Sort eigenvalues and vectors
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    # Calculate angle
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))

    # Scale for confidence interval (df=2 for bivariate normal)
    chi2_val = chi2.ppf(confidence, 2)
    # Axis lenghts (Major and Minor diameters)
    major_axis = 2 * np.sqrt(vals[0] * chi2_val)
    minor_axis = 2 * np.sqrt(vals[1] * chi2_val)

    # Generate points for unrotated centered ellipse
    t = np.linspace(0, 2*np.pi, 100)
    Ell = np.array([major_axis/2 * np.cos(t), minor_axis/2 * np.sin(t)])

    # Rotate
    R = np.array([[np.cos(np.radians(theta)), -np.sin(np.radians(theta))],
                  [np.sin(np.radians(theta)), np.cos(np.radians(theta))]])
    Ell_rot = np.dot(R, Ell)

    # Translate
    return Ell_rot[0, :] + mean_x, Ell_rot[1, :] + mean_y

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("X6 (Product Quality) vs X19 (Customer Satisfaction)",
                                    "X7 (E-Commerce Activities) vs X19 (Customer Satisfaction)"),
                    vertical_spacing=0.15)

# --- Plot 1: X6 vs X19 ---
# 原本是: data_x6 = df_outlier[['X19', 'X6']].dropna()
data_x6 = df[['X19', 'X6']].dropna()  # <--- 將 df_outlier 改成 df
x1 = data_x6['X19']
y1 = data_x6['X6']

fig.add_trace(go.Scatter(x=x1, y=y1, mode='markers', name='Data Points',
                         marker=dict(color='red', size=8)), row=1, col=1)

ell_x1, ell_y1 = get_ellipse_coordinates(x1, y1)
fig.add_trace(go.Scatter(x=ell_x1, y=ell_y1, mode='lines', name='95% Confidence Ellipse',
                         line=dict(color='red')), row=1, col=1)

# --- Plot 2: X7 vs X19 ---
data_x7 = df[['X19', 'X7']].dropna()  # <--- 將 df_outlier 改成 df
x2 = data_x7['X19']
y2 = data_x7['X7']

fig.add_trace(go.Scatter(x=x2, y=y2, mode='markers', name='Data Points',
                         marker=dict(color='red', size=8), showlegend=False), row=2, col=1)

ell_x2, ell_y2 = get_ellipse_coordinates(x2, y2)
fig.add_trace(go.Scatter(x=ell_x2, y=ell_y2, mode='lines', name='95% Confidence Ellipse',
                         line=dict(color='red'), showlegend=False), row=2, col=1)

# Layout adjustments to match style
fig.update_xaxes(title_text="X19 Customer Satisfaction", row=1, col=1)
fig.update_yaxes(title_text="X6 Product Quality", row=1, col=1)
fig.update_xaxes(title_text="X19 Customer Satisfaction", row=2, col=1)
fig.update_yaxes(title_text="X7 E-Commerce Activities", row=2, col=1)

fig.update_layout(height=800, width=700,
                  title_text="Figure 2.10: Selected Scatterplots for Bivariate Detection of Outliers",
                  showlegend=True, template='plotly_white')
fig.show()


## table 2-10(真的很怪)

In [14]:
# [步驟 3-1] 單變量離群值偵測 (Univariate Outlier Detection)
# 原因 (Reason)： 極端值 (Outliers) 會扭曲平均數與相關係數，影響模型準確度。
# 目的 (Purpose)： 將數據標準化為 Z 分數 (Z-score)，並標記絕對值大於 2.5 的樣本為潛在離群值。
# 預期結果 (Result)： 識別出在單一維度上顯著偏離群體的異常樣本 ID。

# 1. 資料準備
# 定義所有相關變數
vars_uni = [f'X{i}' for i in range(6, 20)]
vars_multi = [f'X{i}' for i in range(6, 19)] # Predictors
dep_var = 'X19'

# 讀取最原始的資料表 (不做任何預先刪除)
df_outlier = df.copy()
if 'id' in df_outlier.columns: df_outlier = df_outlier.set_index('id')
elif 'ID' in df_outlier.columns: df_outlier = df_outlier.set_index('ID')

# --- A. Univariate Outliers (Z > 2.5) ---
# 原則：針對每一個變數，使用它自己所有的有效樣本計算 Z-score
uni_results = {}
for col in vars_uni:
    col_data = df_outlier[col].dropna()
    # 使用標準差 (ddof=1, SPSS標準) 計算 Z 分數
    z_scores = np.abs(stats.zscore(col_data, ddof=1))

    # 篩選 > 2.5
    outliers = col_data.index[z_scores > 2.5].tolist()

    if outliers:
        uni_results[col] = str(outliers).replace('[', '').replace(']', '')
    else:
        uni_results[col] = "No cases"

# --- B. Bivariate Outliers (Pairwise Deletion) ---
# 原則：針對每一對 (Xi, X19)，使用兩者均存在的最大樣本計算 D2
bi_results = {}
chi2_threshold = 5.991 # Chi-Square 95% (df=2)

for col in vars_multi:
    # 只取 column 與 dependent variable 的交集樣本
    pair_data = df_outlier[[col, dep_var]].dropna().astype(float)

    # 計算該樣本群的統計量
    cov = pair_data.cov().values
    inv_cov = np.linalg.inv(cov)
    mean = pair_data.mean().values

    d2_values = []
    # 計算距離
    for i, row in pair_data.iterrows():
        d2 = mahalanobis(row.values, mean, inv_cov) ** 2
        d2_values.append(d2)

    # 篩選
    outlier_mask = np.array(d2_values) > chi2_threshold
    outliers = pair_data.index[outlier_mask].tolist()
    outliers.sort()

    # 不做任何人工過濾，這是數據最真實的樣子
    bi_results[col] = str(outliers).replace('[', '').replace(']', '') if outliers else "No cases"


# --- C. Multivariate Outliers (Listwise Mandatory) ---
# 原則：多變量距離必須要求所有變數同時存在，因此 Listwise 是唯一正確做法
# 使用 X6-X18 (Predictors only) 計算
df_multi_data = df_outlier[vars_multi].dropna().astype(float)
cov_m = df_multi_data.cov().values
inv_cov_m = np.linalg.inv(cov_m)
mean_m = df_multi_data.mean().values

d2_list = []
for i, row in df_multi_data.iterrows():
    d2 = mahalanobis(row.values, mean_m, inv_cov_m) ** 2
    d2_list.append(d2)

df_multi_res = pd.DataFrame({'D2': d2_list}, index=df_multi_data.index)
df_multi_res['df'] = len(vars_multi) # 13
df_multi_res['D2/df'] = df_multi_res['D2'] / df_multi_res['df']

# 篩選標準：D2/df > 2.5
multi_outliers = df_multi_res[df_multi_res['D2/df'] > 2.5].sort_values('D2/df', ascending=False)


# --- 4. 格式化輸出表格 ---
rows = []
for var in vars_uni:
    uni_str = uni_results.get(var, "-")
    bi_str = bi_results.get(var, "") if var in bi_results else "" # X19 無 Bivariate

    rows.append({
        'Type': 'Variable Check',
        'Item': var,
        'Univariate Outliers': uni_str,
        'Bivariate Outliers (with X19)': bi_str,
        'Multivariate Outliers': ''
    })

for idx, row in multi_outliers.iterrows():
    rows.append({
        'Type': 'Multivariate Check',
        'Item': str(idx),
        'Univariate Outliers': '',
        'Bivariate Outliers (with X19)': '',
        'Multivariate Outliers': f"D2={row['D2']:.1f}, D2/df={row['D2/df']:.2f}"
    })

df_final_correct = pd.DataFrame(rows)
print("=== Table 2.10 Detection Results (Mathematically/Statistically Correct) ===")
display(df_final_correct.style.set_properties(**{'text-align': 'left'}))


=== Table 2.10 Detection Results (Mathematically/Statistically Correct) ===


,Type,Item,Univariate Outliers,Bivariate Outliers (with X19),Multivariate Outliers
0,Variable Check,X6,No cases,"22, 44, 90",
1,Variable Check,X7,"13, 22, 90","13, 22, 24, 53, 90",
2,Variable Check,X8,87,"22, 87",
3,Variable Check,X9,No cases,"2, 22, 45, 52",
4,Variable Check,X10,No cases,"22, 24, 85",
5,Variable Check,X11,7,"2, 7, 22, 45, 52",
6,Variable Check,X12,90,"22, 44, 90",
7,Variable Check,X13,No cases,"22, 57",
8,Variable Check,X14,77,"22, 77, 84",
9,Variable Check,X15,"6, 53","6, 22, 53",


## table 2-11

In [15]:
# === Table 2.11: Distributional Characteristics & Normality Testing (SPSS Algo + MultiIndex) ===

# 1. 載入完整資料
try:
    df_full = pd.read_excel("data.xls", sheet_name="HBAT", na_values=["NA", "."])
    if 'id' in df_full.columns: df_full = df_full.set_index('id')
    elif 'ID' in df_full.columns: df_full = df_full.set_index('ID')
except:
    df_full = df

# 2. 定義參數
vars_table = [f'X{i}' for i in range(6, 23)]

desc_map = {
    'X6': 'Almost uniform distribution',
    'X7': 'Peaked with positive skew',
    'X8': 'Normal distribution',
    'X9': 'Normal distribution',
    'X10': 'Normal distribution',
    'X11': 'Normal distribution',
    'X12': 'Slight positive skew and peakedness',
    'X13': 'Peaked',
    'X14': 'Normal distribution',
    'X15': 'Normal distribution',
    'X16': 'Negative skewness',
    'X17': 'Peaked, positive skewness',
    'X18': 'Normal distribution',
    'X19': 'Normal distribution',
    'X20': 'Normal distribution',
    'X21': 'Normal distribution',
    'X22': 'Normal distribution',
}

remedy_map = {
    'X6': ('Squared term', lambda x: x**2),
    'X7': ('Logarithm', lambda x: np.log(x)),
    'X13': ('Cubed term', lambda x: x**3),
    'X16': ('Squared term', lambda x: x**2),
    'X17': ('Inverse', lambda x: 1/x)
}

# --- SPSS Dallal-Wilkinson (1986) Algo ---
def spss_lilliefors_p(d_stat, n):
    if d_stat > 0.2: return 0.000
    term1 = -7.01256 * (d_stat**2) * (n + 2.78019)
    term2 = 2.99587 * d_stat * np.sqrt(n + 2.78019)
    term3 = -0.122119 + (0.974598 / np.sqrt(n)) + (1.67997 / n)
    return np.exp(term1 + term2 + term3)

def format_spss_sig(p_val):
    if p_val >= 0.2: return ".200*"
    if p_val < 0.001: return ".000"
    return f"{p_val:.3f}".lstrip('0')

rows = []
print("=== Table 2.11 Final Output (MultiIndex Formatted) ===")

for col in vars_table:
    if col not in df_full.columns: continue
    data = df_full[col].dropna()
    N = len(data)

    # A. Shape Descriptors
    mean = data.mean()
    std = data.std(ddof=1)
    z = (data - mean) / std
    skew_val = (N / ((N-1)*(N-2))) * np.sum(z**3)
    kurt_val = (N*(N+1) / ((N-1)*(N-2)*(N-3))) * np.sum(z**4) - (3*(N-1)**2 / ((N-2)*(N-3)))

    z_skew = skew_val / 0.241
    z_kurt = kurt_val / 0.478

    # B. Normality Test
    ks_stat, _ = lilliefors(data, dist='norm', pvalmethod='table')
    ks_pval = spss_lilliefors_p(ks_stat, N)

    # C. Remedies
    remedy_name = "-"
    remedy_sig = "-"
    if col in remedy_map:
        name, func = remedy_map[col]
        remedy_name = name
        try:
            trans_data = func(data).replace([np.inf, -np.inf], np.nan).dropna()
            rem_d, _ = lilliefors(trans_data, dist='norm', pvalmethod='table')
            remedy_sig = format_spss_sig(spss_lilliefors_p(rem_d, len(trans_data)))
        except:
            remedy_sig = "Err"

    rows.append([
        col,
        f"{skew_val:.3f}", f"{z_skew:.2f}",
        f"{kurt_val:.3f}", f"{z_kurt:.2f}",
        f"{ks_stat:.3f}", format_spss_sig(ks_pval),
        desc_map.get(col, "-"),
        remedy_name,
        remedy_sig
    ])

# Define MultiIndex Columns
columns = pd.MultiIndex.from_tuples([
    ("Variable", "Firm Characteristics", ""),
    ("SHAPE DESCRIPTORS", "Skewness", "Statistic"),
    ("SHAPE DESCRIPTORS", "Skewness", "z value"),
    ("SHAPE DESCRIPTORS", "Kurtosis", "Statistic"),
    ("SHAPE DESCRIPTORS", "Kurtosis", "z value"),
    ("Tests of Normality", "Statistic", ""),
    ("Tests of Normality", "Significance", ""),
    ("Description of the Distribution", "", ""),
    ("Applicable Remedies", "Transformation", ""),
    ("Significance After Remedy", "", "")
])

df_result = pd.DataFrame(rows, columns=columns)

# 為了讓外觀更像書本，可以對不需要顯示的 MultiIndex level 進行一些處理，但在 Pandas 標準顯示中這樣已經很好了
# 我們使用 style 來置中
display(df_result.style.set_properties(**{'text-align': 'center'}))


=== Table 2.11 Final Output (MultiIndex Formatted) ===


## fig 2-14

In [11]:
# === Figure 2.14: Normal Probability Plots (NPP) of Non-normal Metric Variables (Probit Scale) ===

# Variables to plot
vars_to_plot = ['X6', 'X7', 'X12', 'X13', 'X16', 'X17']
var_labels = {
    'X6': 'Product Quality',
    'X7': 'E-Commerce Activities',
    'X12': 'Salesforce Image',
    'X13': 'Competitive Pricing',
    'X16': 'Order & Billing',
    'X17': 'Price Flexibility'
}

# Create Subplots (3 rows x 2 cols)
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"{v} {var_labels[v]}" for v in vars_to_plot],
    vertical_spacing=0.1
)

# Define Probabilities for Y-axis ticks (Pseudo-Probit Scale)
# We plot Z-scores on Y, but label them as Probabilities
tick_probs = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
tick_z = stats.norm.ppf(tick_probs)
tick_text = [f"{p:.2f}" for p in tick_probs]

for i, col_name in enumerate(vars_to_plot):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    series = df[col_name].dropna()
    sorted_data = np.sort(series)
    n = len(sorted_data)
    
    # Calculate observed probabilities (plotting positions)
    probs = (np.arange(1, n + 1) - 0.5) / n
    # Transform observed probabilities to Z-scores (The Probit Transformation)
    # This linearizes the normal CDF.
    # If data is normal, Plot(Value, Z(Prob)) should be a straight line.
    probits = stats.norm.ppf(probs)
    
    # --- Theoretical Line (Straight Line) ---
    # If X ~ Normal(mu, sigma), then Z = (X - mu) / sigma
    # So Z_theoretical = (x_range - mean) / std
    mean_val = series.mean()
    std_val = series.std()
    
    x_range = np.linspace(min(sorted_data), max(sorted_data), 100)
    y_line = (x_range - mean_val) / std_val
    
    # Add Straight Line
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_line,
        mode='lines',
        line=dict(color='red', width=1),
        name='Normal Distribution'
    ), row=row, col=col)

    # Add Data Points (Observed)
    fig.add_trace(go.Scatter(
        x=sorted_data,
        y=probits,
        mode='markers',
        marker=dict(color='red', size=4),
        name='Observed'
    ), row=row, col=col)
    
    # Customize Axis to look like Probability
    fig.update_yaxes(
        tickvals=tick_z,
        ticktext=tick_text,
        range=[min(tick_z), max(tick_z)],
        row=row, col=col
    )

fig.update_layout(
    height=900, width=800,
    title_text="Figure 2.14 Normal Probability Plots (NPP) of Non-normal Metric Variables",
    showlegend=False,
    template='plotly_white'
)
fig.show()


## table 2-12

In [16]:
# === Table 2.12: Testing for Homoscedasticity (Refined Layout) ===

print("=== Table 2.12: Testing for Homoscedasticity ===")

try:
    df_homo = df_full.copy()
except NameError:
    try:
        df_homo = pd.read_excel("data.xls", sheet_name="HBAT", na_values=["NA", "."])
    except Exception:
        df_homo = df.copy()

if 'id' in df_homo.columns:
    df_homo = df_homo.set_index('id')
elif 'ID' in df_homo.columns:
    df_homo = df_homo.set_index('ID')

categorical_map = {
    'X1': 'Customer Type',
    'X2': 'Industry Type',
    'X3': 'Firm Size',
    'X4': 'Region',
    'X5': 'Distribution System'
}

# Use \n for creating visual separation in row labels if supported by style,
# or just Keep standard strings.
metric_blocks = [
    ("Firm Characteristics", [f"X{i}" for i in range(6, 19)]),
    ("Performance Measures", [f"X{i}" for i in range(19, 23)])
]

def format_sig(p):
    if pd.isna(p): return "-"
    if p < 0.001: return ".000"
    formatted = f"{p:.2f}"
    return formatted.lstrip('0') if p < 1 else formatted

def compute_levene(cat_col, metric_col):
    subset = df_homo[[cat_col, metric_col]].dropna()
    if subset.empty: return (np.nan, np.nan)
    groups = [grp[metric_col].values for _, grp in subset.groupby(cat_col)]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2: return (np.nan, np.nan)
    # Defaulting to mean to match likely SPSS default for this analysis
    stat, p_val = levene(*groups, center='mean')
    return stat, p_val

# --- 1. Construct Column MultiIndex (3 Levels) ---
# Level 0: "NONMETRIC/CATEGORICAL VARIABLE" (spanning)
# Level 1: X1..X5
# Level 2: Stat, Sig
column_tuples = []
top_header = "NONMETRIC/CATEGORICAL VARIABLE"

for cat, label in categorical_map.items():
    mid_header = f"{cat} {label}"
    # Tuple structure: (Top, Middle, Bottom)
    column_tuples.append((top_header, mid_header, "Levene Statistic"))
    column_tuples.append((top_header, mid_header, "Sig."))

column_index = pd.MultiIndex.from_tuples(
    column_tuples,
    # The names argument sets the labels for the index levels themselves.
    # The image shows "NONMETRIC..." as a header row, not an index name.
    # So we leave names blank for cleaner look, or set them if we want row headers.
    names=[None, None, None]
)

# --- 2. Construct Rows and Data ---
data_rows = []
row_indices = []

for block_name, metric_vars in metric_blocks:
    for metric in metric_vars:
        if metric not in df_homo.columns: continue

        row_data = []
        for cat, label in categorical_map.items():
            if cat not in df_homo.columns:
                row_data.extend(["-", "-"])
                continue
            stat, p_val = compute_levene(cat, metric)
            row_data.append(f"{stat:.2f}" if pd.notna(stat) else "-")
            row_data.append(format_sig(p_val))

        data_rows.append(row_data)
        row_indices.append((block_name, metric))

# Row Index names: Null for the grouping level, "Metric Variable" for the variable level
row_index = pd.MultiIndex.from_tuples(
    row_indices,
    names=[None, "Metric Variable"]
)

# --- 3. Create DataFrame ---
df_levene = pd.DataFrame(data_rows, index=row_index, columns=column_index)

# --- 4. Styling (Bold p<=.05 & Center Alignment) ---
def highlight_levene(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(0, len(df.columns), 2):
        sig_col_idx = i + 1
        def parse_sig(val):
            try:
                if val == ".000": return 0.0
                if val == "-": return 1.0
                return float(val)
            except:
                return 1.0
        numeric_sigs = df.iloc[:, sig_col_idx].apply(parse_sig)
        mask = numeric_sigs <= 0.05
        # Bold both Stat and Sig columns
        styles.iloc[mask.values, i] = 'font-weight: bold'
        styles.iloc[mask.values, sig_col_idx] = 'font-weight: bold'
    return styles

styled = (df_levene.style
          .apply(lambda x: highlight_levene(df_levene), axis=None)
          .set_properties(**{'text-align': 'center', 'vertical-align': 'middle'})
          .set_table_styles([
              {'selector': 'th', 'props': [('text-align', 'center'), ('vertical-align', 'middle')]},
              # Optional: Try to style the specific headers if possible, but basic center is key
          ]))
display(styled)
print("Notes: Values represent the Levene statistic value and the statistical significance. Values in bold are statistically significant at the .05 level or less.")


=== Table 2.12: Testing for Homoscedasticity ===


Notes: Values represent the Levene statistic value and the statistical significance. Values in bold are statistically significant at the .05 level or less.


## fig 2-15

In [ ]:
# [步驟 4] 變數轉換與常態性檢定 - Figure 2.15
# 原因 (Reason)： 變數 X17 的原始分佈呈現高度偏態，這違反了多變量分析通常要求的「常態性假設」。
# 目的 (Purpose)： 
#   1. 對 X17 進行自然對數轉換 (Log Transform) 試圖修正偏態。
#   2. 計算轉換前後的偏態 (Skewness)、峰度 (Kurtosis) 與 Levene 變異數同質性檢定。
#   3. 繪製並排圖表 (直方圖、常態機率圖、箱型圖) 進行視覺化比較。
# 預期結果 (Result)： 產出 Figure 2.15 Dashboard，證明經對數轉換後，X17 的資料分佈更接近常態，且變異數更為均齊 (Levene 檢定不顯著)。
# --- 1. 資料準備與處理 ---
# 提取原始資料並移除缺失值
x17_orig = df['X17'].dropna()
# 進行自然對數轉換 (Natural Log transformation)
x17_trans = np.log(x17_orig)

# 建立臨時 DataFrame 以方便群組分析
df_temp = df.loc[x17_orig.index].copy()
df_temp['X17_Log'] = x17_trans
# 將 X3 (公司規模) 轉換為具可讀性的標籤
x3_labels = df_temp['X3'].map({0: 'Small (0 to 499)', 1: 'Large (500+)'})

# --- 2. 統計檢定計算 (依照教科書 SPSS 標準) ---
n = len(x17_orig)

# 計算 SPSS 專用的標準誤 (Standard Errors)
ses = np.sqrt((6 * n * (n - 1)) / ((n - 2) * (n + 1) * (n + 3)))
sek = np.sqrt((24 * n * (n - 1)**2) / ((n - 3) * (n - 2) * (n + 3) * (n + 5)))

skew_o = stats.skew(x17_orig, bias=False)
kurt_o = stats.kurtosis(x17_orig, bias=False)
z_skew_o, z_kurt_o = skew_o / ses, kurt_o / sek
ks_stat_o, ks_p_o = lilliefors(x17_orig, dist='norm', pvalmethod='table')

skew_t = stats.skew(x17_trans, bias=False)
kurt_t = stats.kurtosis(x17_trans, bias=False)
z_skew_t, z_kurt_t = skew_t / ses, kurt_t / sek
ks_stat_t, ks_p_t = lilliefors(x17_trans, dist='norm', pvalmethod='table')

levene_vars = {'X1': 'X1 Customer Type', 'X2': 'X2 Industry Type', 'X3': 'X3 Firm Size', 
               'X4': 'X4 Region', 'X5': 'X5 Distribution System'}
levene_data = []

row_o, row_t = ["Original X17"], ["Transformed X17"]

for var, desc in levene_vars.items():
    if var in df_temp.columns:
        # 原始資料檢定
        grps = [df_temp[df_temp[var] == g]['X17'].dropna().values for g in df_temp[var].unique()]
        stat_o, p_o = stats.levene(*grps, center='mean') if len(grps) > 1 else (np.nan, np.nan)
        
        # 轉換後資料檢定
        grps_t = [df_temp[df_temp[var] == g]['X17_Log'].dropna().values for g in df_temp[var].unique()]
        stat_t, p_t = stats.levene(*grps_t, center='mean') if len(grps_t) > 1 else (np.nan, np.nan)
        
        # 格式化輸出 (加上顯著性星號)
        def fmt(v, p): return f"{v:.2f}{'**' if p<.01 else '*' if p<.05 else ''}" if not np.isnan(v) else "-"
        row_o.append(fmt(stat_o, p_o))
        row_t.append(fmt(stat_t, p_t))
    else:
        row_o.append("-"); row_t.append("-")

# --- 3. 繪圖設定 (Plotly) ---
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'xy'}]*3, [{'type': 'xy'}]*3],
    vertical_spacing=0.15, horizontal_spacing=0.08,
    subplot_titles=("Histogram (直方圖)", "Normal Probability Plot (常態機率圖)", "Boxplot (箱型圖)", None, None, None)
)

# 1-1 直方圖與常態曲線
x_range_o = np.linspace(2, 8, 100)
mean_o, std_o = x17_orig.mean(), x17_orig.std()
fig.add_trace(go.Histogram(x=x17_orig, marker_color='#D9534F', xbins=dict(start=2, end=8, size=0.5), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=x_range_o, y=stats.norm.pdf(x_range_o,mean_o,std_o)*len(x17_orig)*0.5, mode='lines', line=dict(color='#333',width=2), showlegend=False), row=1, col=1)
fig.update_xaxes(title_text='Original Distribution of X17', row=1, col=1)

# 1-2 常態機率圖 (NPP)
sorted_o, line_o = np.sort(x17_orig), np.linspace(2, 7.5, 100)
probs = (np.arange(1, n + 1) - 0.5) / n
fig.add_trace(go.Scatter(x=line_o, y=line_o, mode='lines', line=dict(color='#D9534F', width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=sorted_o, y=mean_o+stats.norm.ppf(probs)*std_o, mode='markers', marker=dict(color='#D9534F', size=5, line=dict(color='white', width=0.5)), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text='Observed Value', row=1, col=2); fig.update_yaxes(title_text='Expected Normal Value', row=1, col=2)

# 1-3 箱型圖 (依公司規模分組)
fig.add_trace(go.Box(y=x17_orig, x=x3_labels, boxpoints='outliers', marker=dict(color='#D9534F', size=5), line=dict(color='#333'), showlegend=False), row=1, col=3)
fig.update_xaxes(categoryorder='array', categoryarray=['Small (0 to 499)', 'Large (500+)'], title_text='X3 Firm Size', row=1, col=3)
fig.update_yaxes(title_text='X17 Price Flexibility', row=1, col=3)

mean_t, std_t = x17_trans.mean(), x17_trans.std()
# 2-1 直方圖
x_range_t = np.linspace(0.8, 2.2, 100)
fig.add_trace(go.Histogram(x=x17_trans, marker_color='#5BC0DE', xbins=dict(start=0.8, end=2.2, size=0.15), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=x_range_t, y=stats.norm.pdf(x_range_t,mean_t,std_t)*len(x17_trans)*0.15, mode='lines', line=dict(color='#333',width=2), showlegend=False), row=2, col=1)
fig.update_xaxes(title_text='Transformed Distribution of X17', row=2, col=1)

# 2-2 NPP
sorted_t, line_t = np.sort(x17_trans), np.linspace(0.9, 2.1, 100)
fig.add_trace(go.Scatter(x=line_t, y=line_t, mode='lines', line=dict(color='#5BC0DE', width=2), showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=sorted_t, y=mean_t+stats.norm.ppf(probs)*std_t, mode='markers', marker=dict(color='#5BC0DE', size=5, line=dict(color='white', width=0.5)), showlegend=False), row=2, col=2)
fig.update_xaxes(title_text='Observed Value', row=2, col=2); fig.update_yaxes(title_text='Expected Normal Value', row=2, col=2)

# 2-3 箱型圖
fig.add_trace(go.Box(y=x17_trans, x=x3_labels, boxpoints='outliers', marker=dict(color='#5BC0DE', size=5), line=dict(color='#333'), showlegend=False), row=2, col=3)
fig.update_xaxes(categoryorder='array', categoryarray=['Small (0 to 499)', 'Large (500+)'], title_text='X3 Firm Size', row=2, col=3)
fig.update_yaxes(title_text='Transformed X17', row=2, col=3)

fig.update_layout(height=650, width=1100, title_text="Figure 2.15 Transformation of X17", template='plotly_white', font=dict(family="Helvetica", size=12))
fig.show()

# --- 4. 統計表格輸出 (Pandas DataFrames) ---
print("=== SHAPE DESCRIPTORS (形狀描述統計量) ===")
cols = pd.MultiIndex.from_tuples([
    ("Skewness (偏態)", "Statistic"), ("Skewness (偏態)", "z value"),
    ("Kurtosis (峰度)", "Statistic"), ("Kurtosis (峰度)", "z value"),
    ("Test of Normality (常態性檢定)", "Statistic"), ("Test of Normality (常態性檢定)", "Significance")
])
data_shape = [
    [f"{skew_o:.3f}", f"{z_skew_o:.2f}", f"{kurt_o:.3f}", f"{z_kurt_o:.2f}", f"{ks_stat_o:.3f}", f"{ks_p_o:.3f}"],
    [f"{skew_t:.3f}", f"{z_skew_t:.2f}", f"{kurt_t:.3f}", f"{z_kurt_t:.2f}", f"{ks_stat_t:.3f}", f"{ks_p_t:.3f}"]
]
df_shape = pd.DataFrame(data_shape, columns=cols, index=["Original X17", "Transformed X17"])
df_shape.index.name = "Variable Form"
display(df_shape)

print("\n=== LEVENE TEST STATISTIC (變異數同質性檢定) ===")
df_levene = pd.DataFrame([row_o[1:], row_t[1:]], columns=levene_vars.values(), index=["Original X17", "Transformed X17"])
df_levene.index.name = "Variable Form"
display(df_levene)


=== SHAPE DESCRIPTORS ===


Skewness          Kurtosis         Test of Normality  \
                Statistic z value Statistic z value         Statistic   
Variable Form                                                           
Original X17        0.323    1.34    -0.816   -1.71             0.101   
Transformed X17    -0.121   -0.50    -0.803   -1.68             0.080   

                              
                Significance  
Variable Form                 
Original X17           0.014  
Transformed X17        0.126


=== LEVENE TEST STATISTIC ===


,X1 Customer Type,X2 Industry Type,X3 Firm Size,X4 Region,X5 Distribution System
Variable Form,,,,,
Original X17,5.56**,2.84,4.19*,16.21**,0.62
Transformed X17,2.76,2.23,1.20,3.11,0.01
